# Recovery maps — synthetic losing-reach recoverability (positive control)

Task 10 of the zeta-sensitivity plan: CONUS spatial maps of the synthetic recoverability experiment.

**Inputs**:
- `output/recoverability/recovery_rows.csv` — per-reach results (comid, key_net, key_abs, base_net, base_abs, a_net, a_abs, c_net, c_abs, planted)
- `output/recoverability/plants.csv` — planted sites with staid_nearest
- `output/recoverability/params_orig.nc` — KAN-output Manning's n per COMID (teacher checkpoint, full CONUS)
- `output/recoverability/params_b.nc` — KAN-output Manning's n per COMID (student B after training on planted signal, full CONUS)

**Checkpoint provenance**: teacher = hourly-ON `epoch_5_mb_9` (`.ddrs/runs/2026-07-01T13-43-32Z-train-and-test/checkpoints/epoch_5_mb_9`) + 58 planted synthetic losing sites

**Spec**: `docs/superpowers/specs/2026-07-03-synthetic-recoverability-design.md`

**Generated**: 2026-07-03

In [ ]:
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ── Paths ────────────────────────────────────────────────────────────────────
RECOV_DIR = Path("/home/tbindas/projects/ddrs/output/recoverability")
PLOT_DIR = RECOV_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

RECOVERY_CSV  = RECOV_DIR / "recovery_rows.csv"
PLANTS_CSV    = RECOV_DIR / "plants.csv"
PARAMS_ORIG   = RECOV_DIR / "params_orig.nc"
PARAMS_B      = RECOV_DIR / "params_b.nc"

# Same coordinate source as gradient_maps.ipynb
GAGES_CSV = Path("/home/tbindas/projects/ddr/references/gage_info/gages_3000.csv")
# Same MERIT flowline source as gradient_maps.ipynb
RIV_SHP = Path(
    "/home/tbindas/projects/ddr/data/merit/"
    "riv_pfaf_7_MERIT_Hydro_v07_Basins_v01_bugfix1.shp"
)

XLIM, YLIM = (-125, -66), (24, 53)   # CONUS, EPSG:4326
SAVE_KW = dict(dpi=300, bbox_inches="tight", facecolor="white")

# ── Guard against missing inputs ─────────────────────────────────────────────
for _p in [RECOVERY_CSV, PLANTS_CSV, PARAMS_ORIG, PARAMS_B]:
    if not _p.exists():
        raise FileNotFoundError(
            f"{_p} not found. "
            "Run Task 10 training (synthetic recoverability experiment) to produce it."
        )


def add_positron(ax, crs):
    """CartoDB.Positron basemap; degrades gracefully offline."""
    try:
        cx.add_basemap(
            ax, crs=crs, source=cx.providers.CartoDB.Positron,
            alpha=0.6, zorder=0, attribution=False,
        )
    except Exception as e:
        print(f"basemap skipped ({type(e).__name__}: {e})")
        ax.set_facecolor("#f0f0f0")


def log10_safe(v):
    v = np.asarray(v, dtype=float)
    out = np.full_like(v, np.nan)
    m = np.isfinite(v) & (v > 0)
    out[m] = np.log10(v[m])
    return out


# ── Load recovery rows ────────────────────────────────────────────────────────
df = pd.read_csv(RECOVERY_CSV, dtype={"comid": np.int64})
plants_df = pd.read_csv(PLANTS_CSV, dtype={"comid": np.int64, "staid_nearest": str})
plants_df["staid_nearest"] = plants_df["staid_nearest"].str.zfill(8)

# Filter to planted rows; 'planted' column may be bool or 0/1
planted = df[df["planted"].astype(bool)].copy()
planted = planted.merge(
    plants_df[["comid", "staid_nearest"]], on="comid", how="left"
)
print(f"recovery_rows: {len(df):,} rows; planted sites: {len(planted)}")

# ── Compute recovery ratio ────────────────────────────────────────────────────
planted["recovery_ratio"] = planted["a_net"] / planted["key_net"]
print(
    f"recovery_ratio: median {planted['recovery_ratio'].median():.3f}  "
    f"min {planted['recovery_ratio'].min():.3f}  "
    f"max {planted['recovery_ratio'].max():.3f}"
)

# ── Join gauge lat/lon via staid_nearest (same source as gradient_maps.ipynb) ─
gages = pd.read_csv(GAGES_CSV, dtype={"STAID": str})
gages["STAID"] = gages["STAID"].str.zfill(8)
gages_coord = (
    gages.drop_duplicates("STAID")
    .set_index("STAID")[["LAT_GAGE", "LNG_GAGE"]]
)
planted = planted.join(gages_coord, on="staid_nearest", how="left")
n_missing = planted["LAT_GAGE"].isna().sum()
if n_missing:
    print(f"WARNING: {n_missing} planted sites have no matching gauge coordinate")
print(f"planted sites with coordinates: {(~planted['LAT_GAGE'].isna()).sum()}")

In [ ]:
# ── Panel 1: recovery-ratio gauge scatter (CONUS) ────────────────────────────
has_coords = planted.dropna(subset=["LAT_GAGE", "LNG_GAGE"])
gdf_p = gpd.GeoDataFrame(
    has_coords.reset_index(drop=True),
    geometry=gpd.points_from_xy(has_coords["LNG_GAGE"], has_coords["LAT_GAGE"]),
    crs="EPSG:4326",
)

median_ratio = planted["recovery_ratio"].median()
print(f"R1 median recovery_ratio = {median_ratio:.4f}")

# TwoSlopeNorm: 0.5 = perfect recovery; clip display only, raw values preserved
norm  = mcolors.TwoSlopeNorm(vcenter=0.5, vmin=0.0, vmax=1.5)
cmap  = "RdBu_r"
disp  = gdf_p["recovery_ratio"].clip(0.0, 1.5)   # clip for colormap only

fig, ax = plt.subplots(figsize=(14, 8), dpi=150)
sc = ax.scatter(
    gdf_p.geometry.x,
    gdf_p.geometry.y,
    c=disp,
    cmap=cmap,
    norm=norm,
    s=60,
    zorder=2,
    linewidths=0.6,
    edgecolors="k",
)
add_positron(ax, "EPSG:4326")
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title(
    f"Recovery ratio at planted sites  (R1 median = {median_ratio:.3f}) — "
    "synthetic losing-reach recoverability",
    fontsize=13,
)
cax = make_axes_locatable(ax).append_axes("right", size="3%", pad=0.1)
fig.colorbar(sc, cax=cax).set_label("recovered / planted zeta_net")

out1 = PLOT_DIR / "recovery_ratio_map.png"
fig.savefig(out1, **SAVE_KW)
plt.show()
print(f"saved {out1}")

In [ ]:
# ── Panel 2: Δn absorption map ───────────────────────────────────────────────
ds_orig = xr.open_dataset(PARAMS_ORIG)
ds_b    = xr.open_dataset(PARAMS_B)

# Detect COMID dimension (convention from CLAUDE.md: dim "COMID", full CONUS)
def _comid_dim(ds):
    for d in ds.dims:
        if "comid" in str(d).lower():
            return d
    return list(ds.dims)[0]

dim_orig = _comid_dim(ds_orig)
dim_b    = _comid_dim(ds_b)
comids_orig = ds_orig[dim_orig].values
comids_b    = ds_b[dim_b].values
n_orig = ds_orig["n"].values.astype(np.float64)
n_b    = ds_b["n"].values.astype(np.float64)
print(f"params_orig: {len(comids_orig):,} COMIDs;  params_b: {len(comids_b):,} COMIDs")

# Align on common COMIDs and compute Δn
s_orig = pd.Series(n_orig, index=comids_orig)
s_b    = pd.Series(n_b,    index=comids_b)
common = s_orig.index.intersection(s_b.index)
dn     = (s_b.loc[common] - s_orig.loc[common]).rename("dn")
print(f"common COMIDs: {len(common):,};  |Δn| max {dn.abs().max():.5f}")

# Load MERIT flowlines and join Δn — same technique as gradient_maps.ipynb
gdf_r = gpd.read_file(RIV_SHP, columns=["COMID"]).set_index("COMID")
if gdf_r.crs is None:
    gdf_r = gdf_r.set_crs(epsg=4326)
gdf_r["dn"] = dn.reindex(gdf_r.index).values
gdf_sub = gdf_r.dropna(subset=["dn"]).sort_values("dn")
print(f"joined {len(gdf_sub):,} eval-network reaches of {len(gdf_r):,} fabric flowlines")

vmin_r, vmax_r = np.nanpercentile(gdf_sub["dn"].values, [2, 98])
print(f"Δn  2–98 pct range: [{vmin_r:.5f}, {vmax_r:.5f}]")

fig, ax = plt.subplots(figsize=(14, 8), dpi=150)
gdf_sub.plot(
    ax=ax, column="dn", cmap="plasma_r",
    linewidth=0.4, vmin=vmin_r, vmax=vmax_r, zorder=1,
)
# Overlay planted sites as black x markers
gdf_p.plot(ax=ax, color="black", marker="x", markersize=40, zorder=3, linewidth=1.2)
add_positron(ax, gdf_sub.crs)
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title(
    "Δ Manning's n (B − checkpoint) per MERIT reach — absorption map",
    fontsize=14,
)
cax = make_axes_locatable(ax).append_axes("right", size="3%", pad=0.1)
sm = plt.cm.ScalarMappable(cmap="plasma_r")
sm.set_array([])
sm.set_clim(vmin_r, vmax_r)
fig.colorbar(sm, cax=cax).set_label("Δ Manning's n  (B − checkpoint)")

out2 = PLOT_DIR / "absorption_dn_map.png"
fig.savefig(out2, **SAVE_KW)
plt.show()
print(f"saved {out2}")

In [ ]:
# ── Top-10 worst-recovered planted sites ─────────────────────────────────────
worst = (
    planted.assign(abs_dev=lambda d: (1 - d["recovery_ratio"]).abs())
    .nlargest(10, "abs_dev")
    [["comid", "staid_nearest", "key_net", "a_net", "recovery_ratio"]]
    .rename(columns={"key_net": "target_flux", "a_net": "recovered_flux"})
    .reset_index(drop=True)
)
worst["target_flux"]    = worst["target_flux"].round(4)
worst["recovered_flux"] = worst["recovered_flux"].round(4)
worst["recovery_ratio"] = worst["recovery_ratio"].round(4)
print("Top-10 worst-recovered planted sites (|1 − recovery_ratio| largest):")
print(worst.to_string(index=False))